# Elevator head detector: train and export

This notebook trains a YOLO11n model that finds **heads seen from above**, then exports it as a
`.tflite` file the Elevator app loads as `person-detector.tflite`.

Run the steps **in order**, one cell at a time (click the ▶ on each cell, or press **Ctrl+Enter**).
Read the text above each cell first; it tells you what to expect.

The full walkthrough, including recording and labeling, is in `training/head-detector/GUIDE.md`.

## Step 0: Turn on the GPU

Menu: **Runtime → Change runtime type → T4 GPU → Save**.

Then run the cell below. It should print a line that starts with `GPU 0: Tesla T4`.
If it says `command not found`, the GPU is not on; go back and turn it on.

In [ ]:
!nvidia-smi -L
%pip install -q ultralytics roboflow

## Step 1: Connect Google Drive

Everything is saved to a folder in your Google Drive, so nothing is lost if Colab disconnects.
Run the cell and click **Allow** when Google asks.

In [ ]:
import os
from google.colab import drive

drive.mount('/content/drive')

DRIVE_DIR = '/content/drive/MyDrive/elevator-head-detector'
VIDEO_DIR = f'{DRIVE_DIR}/videos'
FRAME_DIR = f'{DRIVE_DIR}/frames'
RUN_DIR = f'{DRIVE_DIR}/runs'

for folder in (DRIVE_DIR, VIDEO_DIR, FRAME_DIR, RUN_DIR):
    os.makedirs(folder, exist_ok=True)

print('Drive folder ready:', DRIVE_DIR)
print('Put your videos in:  My Drive/elevator-head-detector/videos')

## Step 2: Turn your videos into photos

Before running this, upload your recordings to **My Drive → elevator-head-detector → videos**
(drag them in at drive.google.com).

This cell saves one photo every half second from each video into the `frames` folder, and
makes `frames.zip` next to it. Download that zip from Drive and upload it to Roboflow for labeling
(see the guide).

`EVERY_SECONDS` controls how many photos you get. For 5 minutes of video, 0.5 gives about 600.

In [ ]:
import glob
import shutil
import cv2

EVERY_SECONDS = 0.5
LONGEST_SIDE = 1280

total = 0
for path in sorted(glob.glob(f'{VIDEO_DIR}/*')):
    video = cv2.VideoCapture(path)
    fps = video.get(cv2.CAP_PROP_FPS) or 30
    step = max(1, round(fps * EVERY_SECONDS))
    stem = os.path.splitext(os.path.basename(path))[0]
    index = saved = 0
    while True:
        ok, frame = video.read()
        if not ok:
            break
        if index % step == 0:
            height, width = frame.shape[:2]
            if height > width:
                frame = cv2.rotate(frame, cv2.ROTATE_90_CLOCKWISE)
                height, width = width, height
            scale = LONGEST_SIDE / max(height, width)
            if scale < 1:
                frame = cv2.resize(frame, (round(width * scale), round(height * scale)))
            cv2.imwrite(f'{FRAME_DIR}/{stem}_{index:06d}.jpg', frame, [cv2.IMWRITE_JPEG_QUALITY, 90])
            saved += 1
        index += 1
    print(f'{os.path.basename(path)}: {saved} photos')
    total += saved

print('Total photos:', total)
shutil.make_archive(f'{DRIVE_DIR}/frames', 'zip', FRAME_DIR)
print('Made', f'{DRIVE_DIR}/frames.zip')

## Step 3: Label the photos (outside Colab)

Stop here and follow **Part 3 of the guide** to draw a box around every head in Roboflow.
Come back when you have a dataset **version** generated.

---

## Step 4: Load your labeled dataset

Pick **one** of the two options.

**Option A: from Roboflow (recommended)**

1. In Colab, click the **key icon** (Secrets) on the left sidebar, then **Add new secret**.
   Name it `ROBOFLOW_API_KEY` and paste your key (Roboflow → Settings → API Keys → Private API Key).
   Turn on **Notebook access**.
2. Fill in the three values below. They are in your Roboflow project URL:
   `app.roboflow.com/<WORKSPACE>/<PROJECT>/<VERSION>`
3. Run the cell.

In [ ]:
from google.colab import userdata
from roboflow import Roboflow

WORKSPACE = 'your-workspace'
PROJECT = 'elevator-heads'
VERSION = 1

rf = Roboflow(api_key=userdata.get('ROBOFLOW_API_KEY'))
dataset = rf.workspace(WORKSPACE).project(PROJECT).version(VERSION).download(
    'yolov8', location='/content/dataset', overwrite=True
)
DATASET_DIR = dataset.location
print('Dataset downloaded to', DATASET_DIR)

**Option B: from a zip on Drive** (only if you did not use Option A)

Export from your labeling tool in **YOLO** format, put the zip in the `elevator-head-detector`
Drive folder, set `ZIP_NAME`, and run.

In [ ]:
import shutil

ZIP_NAME = 'dataset.zip'

DATASET_DIR = '/content/dataset'
shutil.rmtree(DATASET_DIR, ignore_errors=True)
shutil.unpack_archive(f'{DRIVE_DIR}/{ZIP_NAME}', DATASET_DIR)
print('Dataset unpacked to', DATASET_DIR)

## Step 5: Check the dataset

This cell fixes the folder paths in `data.yaml` and prints a quick health check:

- **photos and labels per split**: `train` should be the biggest, and `valid` must not be 0
- **heads per photo**: you want a mix of 0, 1, 2, 3 and 4+
- **head size**: compared with the size limits the app uses

If it prints a **WARNING**, fix that before training.

In [ ]:
import yaml
import numpy as np

with open(f'{DATASET_DIR}/data.yaml') as handle:
    original = yaml.safe_load(handle)

names = original.get('names', ['head'])
if isinstance(names, dict):
    names = [names[key] for key in sorted(names)]

splits = {}
for split, folder in (('train', 'train'), ('val', 'valid'), ('test', 'test')):
    if os.path.isdir(f'{DATASET_DIR}/{folder}/images'):
        splits[split] = f'{folder}/images'
if 'val' not in splits and os.path.isdir(f'{DATASET_DIR}/val/images'):
    splits['val'] = 'val/images'

config = {'path': DATASET_DIR, **splits, 'names': names}
DATA_YAML = f'{DATASET_DIR}/data_fixed.yaml'
with open(DATA_YAML, 'w') as handle:
    yaml.safe_dump(config, handle)

print('Classes:', names)
if len(names) != 1:
    print('WARNING: expected exactly one class (head). The app treats a one-class model as a head detector.')

widths, heights, per_image = [], [], []
for split, images in splits.items():
    image_dir = f'{DATASET_DIR}/{images}'
    label_dir = image_dir.replace('/images', '/labels')
    photos = [name for name in os.listdir(image_dir) if name.lower().endswith(('.jpg', '.jpeg', '.png'))]
    labelled = 0
    for photo in photos:
        label_path = f'{label_dir}/{os.path.splitext(photo)[0]}.txt'
        rows = []
        if os.path.exists(label_path):
            rows = [line.split() for line in open(label_path) if line.strip()]
            labelled += 1
        per_image.append(len(rows))
        for row in rows:
            widths.append(float(row[3]))
            heights.append(float(row[4]))
    print(f'{split:5s}: {len(photos)} photos, {labelled} label files')

if 'val' not in splits:
    print('WARNING: no validation split. In Roboflow, generate the version with a Train/Valid/Test split.')

counts = np.bincount(per_image)
print('Heads per photo:', {count: int(amount) for count, amount in enumerate(counts) if amount})
if len(counts) == 0 or counts[0] == 0:
    print('WARNING: no empty photos. Add some frames with nobody in them so the model learns what is not a head.')

if widths:
    print(f'Head width : median {np.median(widths):.3f} of the photo width  (app minimum 0.015)')
    print(f'Head height: median {np.median(heights):.3f} of the photo height (app minimum 0.020)')
    print(f'Largest head area: {max(w * h for w, h in zip(widths, heights)):.3f} of the photo (app maximum 0.15)')

## Step 6: Train

This takes roughly **20–60 minutes** on a T4, depending on how many photos you have.
Progress is saved to Drive after every epoch.

Settings chosen for an overhead camera:

- `degrees=180`, `flipud`, `fliplr`: from above, a head looks the same whichever way the
  person faces, so the photos are spun and flipped to teach that
- `imgsz=640`: heads are small, and 640 keeps enough detail
- `patience=30`: stops early once it stops improving

**If Colab disconnects**, reconnect, run Steps 0, 1, 4 and 5 again, then run the *resume* cell below
instead of this one.

In [ ]:
from ultralytics import YOLO

model = YOLO('yolo11n.pt')
model.train(
    data=DATA_YAML,
    imgsz=640,
    epochs=120,
    patience=30,
    batch=16,
    single_cls=True,
    degrees=180,
    flipud=0.5,
    fliplr=0.5,
    mosaic=1.0,
    close_mosaic=10,
    project=RUN_DIR,
    name='head',
    exist_ok=True,
)

**Resume** (only if training was interrupted):

In [ ]:
from ultralytics import YOLO

YOLO(f'{RUN_DIR}/head/weights/last.pt').train(resume=True)

## Step 7: How good is it?

Two checks:

1. **Detection scores** (`precision`, `recall`, `mAP50`). Aim for recall above 0.9.
2. **Count accuracy**, the number that matters for the elevator. For each test photo, the
   model's head count is compared with your labels, at several confidence cutoffs. It also shows:
   - **under**: counted too few. This is the dangerous one, a tailgater slipping through.
   - **over**: counted too many. That just reopens the door.

Keep a note of these numbers for the paper.

In [ ]:
BEST = f'{RUN_DIR}/head/weights/best.pt'
best = YOLO(BEST)

EVAL_SPLIT = 'test' if 'test' in splits else 'val'
metrics = best.val(data=DATA_YAML, split=EVAL_SPLIT, imgsz=640, verbose=False)
print(f'{EVAL_SPLIT} split -> precision {metrics.box.mp:.3f}  recall {metrics.box.mr:.3f}  mAP50 {metrics.box.map50:.3f}')

image_dir = f'{DATASET_DIR}/{splits[EVAL_SPLIT]}'
label_dir = image_dir.replace('/images', '/labels')
photos = sorted(name for name in os.listdir(image_dir) if name.lower().endswith(('.jpg', '.jpeg', '.png')))

truth = []
for photo in photos:
    label_path = f'{label_dir}/{os.path.splitext(photo)[0]}.txt'
    truth.append(sum(1 for line in open(label_path) if line.strip()) if os.path.exists(label_path) else 0)

scores = []
for photo in photos:
    result = best.predict(f'{image_dir}/{photo}', imgsz=640, conf=0.2, iou=0.5, verbose=False)[0]
    scores.append(result.boxes.conf.cpu().numpy())

print()
print('cutoff   exact   under   over')
summary = []
for cutoff in (0.3, 0.35, 0.4, 0.45, 0.5, 0.55, 0.6, 0.65, 0.7):
    predicted = np.array([int((found >= cutoff).sum()) for found in scores])
    expected = np.array(truth)
    exact = float(np.mean(predicted == expected))
    under = float(np.mean(predicted < expected))
    over = float(np.mean(predicted > expected))
    summary.append((cutoff, exact, under, over))
    print(f'{cutoff:5.2f}   {exact:6.1%}  {under:6.1%}  {over:6.1%}')

choice = max(summary, key=lambda row: (row[1] - 2 * row[2], -row[0]))
print()
print(f'Suggested app setting: PERSON_SCOPE_HEAD.minScore = {choice[0]:.2f}')
print('(chosen for the most exact counts, with under-counting weighted twice as bad)')

**Look at some predictions.** Green boxes are what the model found. Check that bending people,
people close together and people with hats all get their own box.

In [ ]:
import glob
from IPython.display import Image, display

samples = [f'{image_dir}/{photo}' for photo in photos[:8]]
results = best.predict(samples, imgsz=640, conf=choice[0], iou=0.5, save=True,
                       project='/content/preview', name='heads', exist_ok=True, verbose=False)
for path in sorted(glob.glob('/content/preview/heads/*.jpg'))[:8]:
    display(Image(filename=path, width=640))

## Step 8: Export for the phone

This makes the `.tflite` file, saves a copy in Drive as `person-detector.tflite`, and downloads it.

If the download does not start, get it from **My Drive → elevator-head-detector**.

In [ ]:
import shutil
from google.colab import files

exported = best.export(format='tflite', imgsz=640)
target = f'{DRIVE_DIR}/person-detector.tflite'
shutil.copy(exported, target)
print('Saved to Drive:', target)
files.download(target)

## Step 9: Check the exported file the way the app will use it

This runs the **exported `.tflite`** (not the training file) on the test photos, preparing each photo
exactly like the app does: fit inside a square, black bars, colours scaled to 0–1. If its count
accuracy is close to Step 7, the export is good.

The shapes printed first should be an input of `[1, 3, 640, 640]` (or `[1, 640, 640, 3]`) and an
output of `[1, 5, 8400]`. The app reads that single-class output as a head detector.

In [ ]:
import cv2

try:
    from ai_edge_litert.interpreter import Interpreter
except ImportError:
    from tensorflow.lite.python.interpreter import Interpreter

interpreter = Interpreter(model_path=target)
interpreter.allocate_tensors()
input_info = interpreter.get_input_details()[0]
output_info = interpreter.get_output_details()[0]
print('input ', list(input_info['shape']), input_info['dtype'].__name__)
print('output', list(output_info['shape']), output_info['dtype'].__name__)

channels_first = input_info['shape'][1] == 3
size = int(input_info['shape'][2] if channels_first else input_info['shape'][1])


def prepare(path):
    image = cv2.cvtColor(cv2.imread(path), cv2.COLOR_BGR2RGB)
    height, width = image.shape[:2]
    scale = size / max(height, width)
    resized = cv2.resize(image, (max(1, round(width * scale)), max(1, round(height * scale))))
    canvas = np.zeros((size, size, 3), dtype=np.float32)
    pad_y = (size - resized.shape[0]) // 2
    pad_x = (size - resized.shape[1]) // 2
    canvas[pad_y:pad_y + resized.shape[0], pad_x:pad_x + resized.shape[1]] = resized / 255.0
    tensor = canvas.transpose(2, 0, 1)[None] if channels_first else canvas[None]
    return tensor.astype(np.float32)


def iou(a, b):
    left, top = max(a[0], b[0]), max(a[1], b[1])
    right, bottom = min(a[2], b[2]), min(a[3], b[3])
    overlap = max(0.0, right - left) * max(0.0, bottom - top)
    union = (a[2] - a[0]) * (a[3] - a[1]) + (b[2] - b[0]) * (b[3] - b[1]) - overlap
    return overlap / union if union > 0 else 0.0


def count_heads(path, cutoff):
    interpreter.set_tensor(input_info['index'], prepare(path))
    interpreter.invoke()
    raw = interpreter.get_tensor(output_info['index'])[0]
    if raw.shape[0] > raw.shape[1]:
        raw = raw.T
    boxes = []
    for anchor in np.where(raw[4] >= cutoff)[0]:
        cx, cy, w, h = raw[0:4, anchor]
        boxes.append((float(raw[4, anchor]), (cx - w / 2, cy - h / 2, cx + w / 2, cy + h / 2)))
    boxes.sort(key=lambda item: -item[0])
    kept = []
    for score, box in boxes:
        if all(iou(box, other) < 0.5 for other in kept):
            kept.append(box)
    return len(kept)


predicted = np.array([count_heads(f'{image_dir}/{photo}', choice[0]) for photo in photos])
expected = np.array(truth)
print()
print(f'Exported model at cutoff {choice[0]:.2f}: exact {np.mean(predicted == expected):.1%}, '
      f'under {np.mean(predicted < expected):.1%}, over {np.mean(predicted > expected):.1%}')

## Done

Next steps are in **Part 5 of the guide**: put `person-detector.tflite` in `src/assets/models/`,
run `npm run verify-person-model` (it should say `yolo (head detector)`), set
`PERSON_SCOPE_HEAD.minScore` to the suggested value, and rebuild the app.